In [1]:
import pandas as pd
import numpy as np
import os

# Sensor channels from the paper
SENSOR_CHANNELS = ['acc', 'acoustic_emission', 'force_x', 'force_y', 'force_z']
N_SENSOR_CHANNELS = len(SENSOR_CHANNELS)  # 5
SAMPLE_RATE = 1626  # Hz from the paper

def load_sensor_csv(csv_path):
    """
    Load one sensor CSV for a single milling pass.

    Returns:
        np.array of shape (N_timesteps, 5)
    """
    df = pd.read_csv(csv_path, header=None)

    # Select only the 5 sensor channels (fixed: was 0:4, missing force_z)
    data = df.iloc[:, 0:N_SENSOR_CHANNELS].values.astype(np.float32)

    print(f"Loaded: {data.shape}")  # e.g., (94000, 5)
    return data


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import joblib

def fit_pca_on_training_data(train_csv_paths, n_components=None, 
                              variance_threshold=0.95):
    """
    Fits PCA on all training sensor data combined.
    
    IMPORTANT: PCA must be fit ONLY on training data to avoid leakage.
    
    Args:
        train_csv_paths: list of CSV paths from training sets only
        n_components: fixed number of components (None = use variance threshold)
        variance_threshold: keep enough components to explain this much variance
    
    Returns:
        fitted scaler, fitted PCA, chosen n_components
    """
    print("Loading all training sensor data for PCA fitting...")
    
    all_data = []
    for csv_path in train_csv_paths:
        data = load_sensor_csv(csv_path)  # (N_timesteps, 5)
        all_data.append(data)
    
    # Stack all training data: (total_timesteps, 5)
    all_data = np.vstack(all_data)
    print(f"Total training timesteps for PCA: {all_data.shape}")
    
    # Step 1: StandardScaler - CRITICAL before PCA
    # PCA is sensitive to scale - sensors have very different units
    # acc is in g, force in Newtons, AE in volts - must normalize first
    scaler = StandardScaler()
    all_data_scaled = scaler.fit_transform(all_data)
    
    # Step 2: Fit PCA
    if n_components is None:
        # First fit with all components to find optimal number
        pca_full = PCA(n_components=None)
        pca_full.fit(all_data_scaled)
        
        # Find number of components to explain variance_threshold
        cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
        n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        
        print(f"\nVariance explained by each component:")
        for i, (var, cum_var) in enumerate(
            zip(pca_full.explained_variance_ratio_, cumulative_variance)
        ):
            print(f"  PC{i+1}: {var:.4f} ({cum_var:.4f} cumulative)")
            if cum_var >= variance_threshold:
                print(f"  → {n_components} components explain "
                      f"{cum_var:.2%} of variance")
                break
        
        # Plot explained variance
        plot_pca_variance(pca_full.explained_variance_ratio_)
    
    # Step 3: Fit final PCA with chosen n_components
    pca = PCA(n_components=n_components)
    pca.fit(all_data_scaled)
    
    print(f"\nFinal PCA: {n_components} components")
    print(f"Total variance explained: "
          f"{pca.explained_variance_ratio_.sum():.4f}")
    
    # Save scaler and PCA for later use on val/test data
    joblib.dump(scaler, 'scaler.pkl')
    joblib.dump(pca,    'pca.pkl')
    
    return scaler, pca, n_components


def plot_pca_variance(explained_variance_ratio):
    """Visualize how many components are needed"""
    cumulative = np.cumsum(explained_variance_ratio)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Individual variance per component
    axes[0].bar(range(1, len(explained_variance_ratio) + 1), 
                explained_variance_ratio)
    axes[0].set_xlabel('Principal Component')
    axes[0].set_ylabel('Explained Variance Ratio')
    axes[0].set_title('Variance per Component')
    
    # Cumulative variance
    axes[1].plot(range(1, len(cumulative) + 1), cumulative, 'bo-')
    axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
    axes[1].axhline(y=0.99, color='g', linestyle='--', label='99% threshold')
    axes[1].set_xlabel('Number of Components')
    axes[1].set_ylabel('Cumulative Explained Variance')
    axes[1].set_title('Cumulative Variance Explained')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('pca_variance.png')
    plt.show()


def apply_pca_to_pass(csv_path, scaler, pca):
    """
    Apply fitted scaler + PCA to a single milling pass.
    
    Returns:
        np.array of shape (N_timesteps, n_components)
    """
    data = load_sensor_csv(csv_path)         # (N_timesteps, 5)
    data_scaled = scaler.transform(data)     # (N_timesteps, 5) - scaled
    data_pca = pca.transform(data_scaled)    # (N_timesteps, n_components)
    
    return data_pca.astype(np.float32)

Single scalar wear label per pass (broadcast removed)

In [3]:
def build_pass_label(wear_value_um):
    """
    Returns a single normalised wear scalar for one milling pass.
    Wear is measured once per pass, so one label per pass is correct.

    Args:
        wear_value_um: scalar wear in micrometers (e.g., 187.0)

    Returns:
        float in [0, 1]
    """
    MAX_WEAR_UM = 450.0
    return np.float32(wear_value_um / MAX_WEAR_UM)


#### Window creation (one scalar label per window)

In [4]:
def create_windows(sensor_data_pca, wear_label_scalar,
                   window_size=500, stride=250):
    """
    Splits a long time series into overlapping windows.
    Each window gets a single scalar wear label (not per-timestep).

    Args:
        sensor_data_pca:   (N_timesteps, n_components)
        wear_label_scalar: float — the wear value for this entire pass
        window_size:       number of timesteps per window
        stride:            step between windows

    Returns:
        windows: (N_windows, window_size, n_components)
        labels:  (N_windows,)  — one scalar per window
    """
    n_timesteps = sensor_data_pca.shape[0]

    windows = []
    labels  = []

    for start in range(0, n_timesteps - window_size + 1, stride):
        end = start + window_size
        windows.append(sensor_data_pca[start:end, :])  # (window_size, n_components)
        labels.append(wear_label_scalar)               # scalar repeated per window

    windows = np.array(windows, dtype=np.float32)      # (N_windows, window_size, n_components)
    labels  = np.array(labels,  dtype=np.float32)      # (N_windows,)

    print(f"Created {len(windows)} windows from {n_timesteps} timesteps")
    print(f"Windows shape: {windows.shape}")

    return windows, labels


#### Dataset Class

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class WearRegressionDataset(Dataset):
    def __init__(self, index_df, scaler, pca,
                 window_size=500, stride=250):
        """
        Args:
            index_df:    DataFrame with columns: sensor_path, wear_value
            scaler:      fitted StandardScaler (training data only)
            pca:         fitted PCA (training data only)
            window_size: timesteps per LSTM input window
            stride:      step between windows
        """
        self.window_size = window_size
        self.stride      = stride

        self.all_windows = []
        self.all_labels  = []

        print("Preprocessing dataset...")
        for _, row in index_df.iterrows():

            # Apply PCA to sensor data
            sensor_pca = apply_pca_to_pass(
                row['sensor_path'], scaler, pca
            )  # (N_timesteps, n_components)

            # Single normalised label for this pass
            label = build_pass_label(row['wear_value'])  # scalar float32

            # Create windows — label broadcast inside create_windows
            windows, win_labels = create_windows(
                sensor_pca, label, window_size, stride
            )  # (N_windows, window_size, k), (N_windows,)

            self.all_windows.append(windows)
            self.all_labels.append(win_labels)

        self.all_windows = np.vstack(self.all_windows)   # (total_windows, window_size, k)
        self.all_labels  = np.concatenate(self.all_labels)  # (total_windows,)

        print(f"Total windows in dataset: {self.all_windows.shape[0]}")

    def __len__(self):
        return len(self.all_windows)

    def __getitem__(self, idx):
        window = self.all_windows[idx]  # (window_size, n_components)
        label  = self.all_labels[idx]   # scalar

        return {
            'sensor': torch.tensor(window, dtype=torch.float32),
            'wear':   torch.tensor(label,  dtype=torch.float32)  # scalar
        }


#### Declaring LSTM Model 

In [6]:
import torch
import torch.nn as nn

class WearLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128,
                 num_layers=2, dropout=0.3):
        """
        LSTM for single wear value regression per window.
        Uses the last hidden state (not per-timestep output).

        Args:
            input_size:  number of PCA components (k)
            hidden_size: LSTM hidden state dimension
            num_layers:  number of stacked LSTM layers
            dropout:     dropout between LSTM layers
        """
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # Bidirectional doubles hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()   # Output in [0, 1] to match normalised wear
        )

    def forward(self, x):
        """
        Args:
            x: (batch, window_size, n_components)

        Returns:
            wear: (batch,) — single predicted wear value per window
        """
        lstm_out, _ = self.lstm(x)  # (batch, window_size, hidden_size * 2)

        # Use last timestep's output to summarise the window
        last_out = lstm_out[:, -1, :]  # (batch, hidden_size * 2)

        wear = self.regressor(last_out)  # (batch, 1)
        return wear.squeeze(-1)          # (batch,)


#### Training Loop

In [7]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
from tqdm import tqdm

def train_wear_lstm(train_loader, val_loader, n_components,
                    num_epochs=20, device='cuda'):

    model = WearLSTM(
        input_size=n_components,
        hidden_size=128,
        num_layers=2,
        dropout=0.3
    ).to(device)

    criterion = nn.MSELoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    scheduler = ReduceLROnPlateau(
        optimizer, mode='min',
        factor=0.5, patience=5
    )

    best_val_loss = float('inf')
    train_losses, val_losses = [], []

    epoch_bar = tqdm(range(num_epochs), desc='Epochs', unit='epoch')


    for epoch in epoch_bar:

        # -------- Training --------
        model.train()
        epoch_train_loss = 0.0

        train_bar = tqdm(train_loader, desc=f'  Train', leave=False, unit='batch')
        for batch in train_bar:
            sensor = batch['sensor'].to(device)  # (batch, window_size, k)
            wear   = batch['wear'].to(device)    # (batch,) — scalar per window

            optimizer.zero_grad()
            predictions = model(sensor)          # (batch,)
            loss = criterion(predictions, wear)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_train_loss += loss.item()
            train_bar.set_postfix(loss=f'{loss.item():.6f}')

        avg_train_loss = epoch_train_loss / len(train_loader)

        # -------- Validation --------
        model.eval()
        epoch_val_loss = 0.0
        all_preds, all_targets = [], []

        val_bar = tqdm(val_loader, desc=f'  Val  ', leave=False, unit='batch')
        with torch.no_grad():
            for batch in val_bar:
                sensor = batch['sensor'].to(device)
                wear   = batch['wear'].to(device)

                predictions = model(sensor)      # (batch,)
                loss = criterion(predictions, wear)

                epoch_val_loss += loss.item()
                all_preds.extend(predictions.cpu().numpy().tolist())
                all_targets.extend(wear.cpu().numpy().tolist())
                val_bar.set_postfix(loss=f'{loss.item():.6f}')

        avg_val_loss = epoch_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)

        # MAE in original µm scale
        all_preds   = np.array(all_preds)   * 450.0
        all_targets = np.array(all_targets) * 450.0
        mae_um = np.mean(np.abs(all_preds - all_targets))

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        # Update epoch bar with summary stats
        current_lr = optimizer.param_groups[0]['lr']
        epoch_bar.set_postfix(
            train=f'{avg_train_loss:.6f}',
            val=f'{avg_val_loss:.6f}',
            mae=f'{mae_um:.2f}µm',
            lr=f'{current_lr:.2e}'
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'val_mae_um':  mae_um
            }, 'best_wear_lstm.pth')
            tqdm.write(f'  ✓ Epoch {epoch+1}: saved best model (MAE: {mae_um:.2f}µm)')

    return model, train_losses, val_losses


#### Entire pipeline put together

In [ ]:
import glob
import pandas as pd
from pathlib import Path

# ---- 1. Load labels.csv ----
labels_df = pd.read_csv('E:\DF\MATWI\MATWI\labels.csv')

print(f"labels.csv shape: {labels_df.shape}")
print(f"Columns: {labels_df.columns.tolist()}")
print(f"NaN count in SensorFile: {labels_df['SensorFile'].isna().sum()}")
print(f"NaN count in wear:       {labels_df['wear'].isna().sum()}")

# Drop rows where SensorFile or wear is missing
labels_df = labels_df.dropna(subset=['SensorFile', 'wear'])
print(f"After dropping NaNs: {len(labels_df)} rows remain")

# Build lookup: sensor CSV filename → wear value (µm)
wear_map = {
    Path(str(row['SensorFile'])).name: float(row['wear'])
    for _, row in labels_df.iterrows()
}

print(f"\nLoaded {len(wear_map)} wear labels")
print(f"Sample entries: {list(wear_map.items())[:3]}")


# ---- 2. Collect CSV paths per set ----
all_sets = {}
for i in range(1, 18):
    all_sets[f'Set{i}'] = sorted(glob.glob(f'E:\DF\MATWI\MATWI\Set{i}\sensordata\*.csv'))

TRAIN_SETS = ['Set1','Set2','Set5','Set7','Set8','Set10','Set11']
VAL_SETS   = ['Set3','Set6','Set12']
TEST_SETS  = ['Set4','Set9','Set13']


# ---- 3. Build index DataFrames ----
def build_index_df(set_names):
    """
    Match every sensor CSV to its wear label from labels.csv.
    Returns a DataFrame with columns: sensor_path, wear_value, set.
    """
    rows = []
    for set_name in set_names:
        for csv_path in all_sets[set_name]:
            fname = Path(csv_path).name
            if fname in wear_map:
                rows.append({
                    'sensor_path': csv_path,
                    'wear_value':  wear_map[fname],
                    'set':         set_name
                })
            else:
                print(f"WARNING: no wear label for {fname} — skipping.")
    return pd.DataFrame(rows)


train_df = build_index_df(TRAIN_SETS)
val_df   = build_index_df(VAL_SETS)
test_df  = build_index_df(TEST_SETS)

print(f"\nTrain: {len(train_df)} passes")
print(f"Val:   {len(val_df)} passes")
print(f"Test:  {len(test_df)} passes")
print(f"\nWear range in train: {train_df['wear_value'].min():.1f} – {train_df['wear_value'].max():.1f} µm")
print(train_df.head())


# ---- 4. Fit PCA on training sensor data only ----
train_csv_paths = train_df['sensor_path'].tolist()

scaler, pca, n_components = fit_pca_on_training_data(
    train_csv_paths,
    variance_threshold=0.95
)


labels.csv shape: (1803, 11)
Columns: ['ImageName', 'SensorName', 'Set', 'ImageID', 'SensorID', 'wear', 'type', 'ImageDateTime', 'SensorDateTime', 'ImageFile', 'SensorFile']
NaN count in SensorFile: 103
NaN count in wear:       140
After dropping NaNs: 1566 rows remain

Loaded 1566 wear labels
Sample entries: [('File_name_2022-09-09T13_30_37.534347.csv', 30.0), ('File_name_2022-09-09T13_42_22.323924.csv', 30.0), ('File_name_2022-09-09T13_57_28.734803.csv', 60.0)]

Train: 0 passes
Val:   0 passes
Test:  0 passes


KeyError: 'wear_value'

In [ ]:
# Apply scalar+PCA to val and test datasets

train_dataset = WearRegressionDataset(
    train_df, scaler, pca, window_size=500, stride=250
)
val_dataset = WearRegressionDataset(
    val_df, scaler, pca, window_size=500, stride=500  # No overlap for eval
)
test_dataset = WearRegressionDataset(
    test_df, scaler, pca, window_size=500, stride=500
)


# ---- 5. Create DataLoaders ----
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,  num_workers=0
)
val_loader = DataLoader(
    val_dataset,   batch_size=32, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset,  batch_size=32, shuffle=False, num_workers=0
)

# ---- 6. Train ----
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model, train_losses, val_losses = train_wear_lstm(
    train_loader, val_loader,
    n_components=n_components,
    num_epochs=20,
    device=device
)

Preprocessing dataset...
Loaded: (89000, 5)
Created 355 windows from 89000 timesteps
Windows shape: (355, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (75000, 5)
Created 299 windows from 75000 timesteps
Windows shape: (299, 500, 4)
Loaded: (75000, 5)
Created 299 windows from 75000 timesteps
Windows shape: (299, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (76000, 5)
Created 303 windows from 76000 timesteps
Windows shape: (303, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500, 4)
Loaded: (78000, 5)
Created 311 windows from 78000 timesteps
Windows shape: (311, 500,

Epochs:   2%|▏         | 1/50 [06:05<4:58:10, 365.12s/epoch, lr=1.00e-03, mae=67.31µm, train=0.020629, val=0.056407]

  ✓ Epoch 1: saved best model (MAE: 67.31µm)


Epochs:  14%|█▍        | 7/50 [36:23<3:28:45, 291.29s/epoch, lr=1.00e-03, mae=74.57µm, train=0.007423, val=0.054186]

  ✓ Epoch 7: saved best model (MAE: 74.57µm)


Epochs:  38%|███▊      | 19/50 [1:31:17<2:21:55, 274.69s/epoch, lr=5.00e-04, mae=72.11µm, train=0.004831, val=0.052128]

  ✓ Epoch 19: saved best model (MAE: 72.11µm)


Epochs: 100%|██████████| 50/50 [3:57:19<00:00, 284.79s/epoch, lr=1.56e-05, mae=75.47µm, train=0.002934, val=0.059738]  
